In [ ]:
import os, sys, glob, json, shutil, subprocess, time
T0 = time.time()
MODE = 'full'
NSH = 15
MAX_HOURS = 11.3
ARGS = ['--arms', 'E_prior', '--backbone', 'tf_efficientnetv2_s.in21k_ft_in1k', '--init', 'timm', '--folds', '10', '--only-folds', '3', '--epochs', '16', '--batch', '2', '--train-windows', '64', '--crop', '224', '--slices', '3', '--phys-mm', '3.5', '--spatial-attn', '--lr', '1e-4', '--head-lr', '1e-3', '--wd', '0.05', '--workers', '4', '--eval-workers', '2', '--eval-stride', '3', '--resume', '--ckpt-every', '1', '--i-have-user-authorisation-to-finetune']
TARGETS_FILE = 'targets_b_v5h.csv'
def shallow(name):   # never recurse into the competition's DICOM tree
    hits = []
    for pat in ('/kaggle/input/*/', '/kaggle/input/*/*/', '/kaggle/input/*/*/*/', '/kaggle/input/*/*/*/*/'):
        hits += [h for h in glob.glob(pat + name) if 'abnormality-detection' not in h]
    return sorted(set(hits))
shards = sorted({os.path.dirname(p) for p in shallow('px_s*/index.json')})
print('shards found', len(shards), [os.path.basename(s) for s in shards], flush=True)
if MODE == 'full':
    assert len(shards) == NSH, 'need all %d shards' % NSH
kit = os.path.dirname(shallow('compact_train_shards.py')[0])
TARGETS_PATH = shallow(TARGETS_FILE)[0] if TARGETS_FILE else kit + '/targets_b_v3.csv'
print('targets', TARGETS_PATH, flush=True)
OUT = '/kaggle/working/b_out'
prev = shallow('b_out/ckpt')
if prev and not os.path.isdir(OUT):
    shutil.copytree(os.path.dirname(prev[0]), OUT)
    print('RESUME from', os.path.dirname(prev[0]), os.listdir(OUT + '/ckpt'), flush=True)
os.makedirs(OUT, exist_ok=True)
env = dict(os.environ, CT_FP16='1', PYTHONUNBUFFERED='1')
cmd = [sys.executable, '-u', kit + '/compact_train_shards.py', '--px', ','.join(shards), '--targets', TARGETS_PATH,
       '--out', OUT] + ARGS + (['--forward-only'] if MODE == 'smoke' else [])
print(' '.join(cmd), flush=True)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=False)
MAX_SEC = MAX_HOURS * 3600   # stop before the account's GPU quota runs out; the fold checkpoint stays in b_out/ckpt
try:
    rc = subprocess.run(cmd, env=env, timeout=MAX_SEC).returncode
except subprocess.TimeoutExpired:
    rc = 0
    print('SESSION_TIMEBOX %.1f h reached: resume from b_out/ckpt in the next session' % MAX_HOURS, flush=True)
print('TRAIN_RC', rc, 'minutes %.1f' % ((time.time() - T0) / 60), flush=True)
for p in sorted(glob.glob(OUT + '/**', recursive=True))[:60]:
    if os.path.isfile(p):
        print('  out', p[len(OUT) + 1:], os.path.getsize(p), flush=True)
if rc != 0:
    raise SystemExit(rc)
